# CP0 · Examen blanc de la remise à niveau — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/checkpoints/partie_0/03_examen_solutions.ipynb)

L'enregistrement des réponses que vérifie le notebook de l'examen (cellules marquées `answer`, récoltées par `tools/build_answers.py`), calculées à partir des nombres du sujet ; les erreurs classiques reçoivent un message qui dit où chercher. Le corrigé détaillé et le barème sont dans `03_examen_corrige.md`.

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast

🧪 workbookMaths : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.18 | numpy 2.1.3 | scipy 1.16.3 | sympy 1.14.0 | matplotlib 3.10.0 | pandas 2.2.3 | scikit-learn 1.6.1
   Graine : 42 | FAST_MODE : True


## Les réponses vérifiées après l'examen

In [2]:
# Part B: the answers are computed from the numbers of the statements (01_examen_sujet.md)
import math
from fractions import Fraction

import sympy as sp

P_C, P_V = 0.7, 0.3                       # CP0.7: share of the parcels sent by van and by bike
LATE_C, LATE_V = 0.04, 0.08               # late rate of each way
P_R = P_C * LATE_C + P_V * LATE_V         # probability of a late parcel (total probability)
E_X = (1 - P_R) * 2 + P_R * (-10)         # gain: +2 for a parcel on time, -10 for a late one
V_X = (1 - P_R) * 2 ** 2 + P_R * (-10) ** 2 - E_X ** 2

**CP0.1 — Fractions, puissances et notation scientifique**

In [3]:
wbm.record("CP0.1a", Fraction(7, 4) - Fraction(5, 6) / Fraction(4, 9), kind="fraction", simplified=True, mistakes={"la division passe avant la soustraction (les priorités)": "33/16", "diviser par une fraction, c'est multiplier par son inverse": "149/108"})
wbm.record("CP0.1b", Fraction(5) ** 4 * Fraction(25) ** -3 / Fraction(5) ** -1, kind="fraction", simplified=True, mistakes={"écris tout avec des puissances de 5 : 25 est une puissance de 5": "25", "diviser par une puissance d'exposant négatif, c'est multiplier par la puissance opposée": "1/125"})
wbm.record("CP0.1c", [2.8, 4], decimals=1, mistakes={"en notation scientifique, 1 ≤ a < 10": [28, 3], "multiplier des puissances de 10, c'est additionner leurs exposants": [2.8, -40]})
wbm.record("CP0.1d", [6, 3], decimals=1, mistakes={"diviser des puissances de 10, c'est soustraire les exposants, avec leurs signes": [6, -11], "on soustrait l'exposant du dénominateur à celui du numérateur, pas l'inverse": [6, -3], "en notation scientifique, 1 ≤ a < 10": [0.6, 4]})
wbm.record("CP0.1e", -10.0, decimals=1, mistakes={"des évolutions successives ne s'additionnent pas : multiplie leurs coefficients": -5, "c'est le coefficient multiplicateur ; l'énoncé demande l'évolution en pourcentage": 0.9, "c'est l'évolution en proportion ; l'énoncé la demande en pourcentage": -0.1})

WBM_ANSWER {"hash": "342814dd156d404d6d773aa11d97343ebcf1b62f0ada406f4dd6b6df1cd654da", "id": "CP0.1a", "kind": "fraction", "mistakes": {"b17227221632b8dfd27ca5c0fdf7da7006714a2e05f3658863c9244ffbd9b6c8": "diviser par une fraction, c'est multiplier par son inverse", "ce63192f30cf208262e0327abb251c8f46d03f17ad11e0e96d05f422a55497c4": "la division passe avant la soustraction (les priorités)"}, "sign": -1, "simplified": true}
📝 Ex CP0.1a : réponse enregistrée (fraction).
WBM_ANSWER {"hash": "c18c92e8f170c4e60d052ef1947bb0fb6db6bc025248401531ea06b89da8efd5", "id": "CP0.1b", "kind": "fraction", "mistakes": {"911cae175df195c6b6a67a773afccc19eaa476285259127577b89e81d505129c": "diviser par une puissance d'exposant négatif, c'est multiplier par la puissance opposée", "ddd8ee4eb2e8c8d2c946bc41ec81d36104d58b330524b3df4f9f8c47c8e200bf": "écris tout avec des puissances de 5 : 25 est une puissance de 5"}, "sign": 1, "simplified": true}
📝 Ex CP0.1b : réponse enregistrée (fraction).
WBM_ANSWER {"decim

**CP0.2 — Développer, factoriser, résoudre**

In [4]:
wbm.record("CP0.2a", "8*x**2 + 2*x - 5", kind="expr", variables=("x",), expanded=True, mistakes={"un signe moins devant une parenthèse change tous les signes qu'elle contient": "8*x**2 - 2*x - 3", "(a − b)² = a² − 2ab + b²": "8*x**2 - 5"})
wbm.record("CP0.2b", [0, 7], mistakes={"diviser les deux membres par x fait perdre une solution : factorise": [7]})
wbm.record("CP0.2c", Fraction(-3, 8), kind="fraction", simplified=True, mistakes={"développe −3(2x − 1) en gardant les deux signes : (−3) × (−1) = +3": "-9/8"})
wbm.record("CP0.2d", "]-oo ; -3[", kind="interval", mistakes={"diviser par un nombre négatif change le sens de l'inégalité": "]-3 ; +oo[", "une inégalité stricte n'atteint pas sa borne : crochet ouvert": "]-oo ; -3]"})
wbm.record("CP0.2e", "[-1 ; 7]", kind="interval", mistakes={"|x − 3| est la distance de x à 3 : les solutions sont autour de 3": "[-7 ; 1]", "avec ≤, les bornes sont atteintes : crochets fermés": "]-1 ; 7["})

WBM_ANSWER {"expanded": true, "hash": "67cd3a7f3ded8d6f160c2feecd09e133c78d8c6c09e025722f39b1d044181b1a", "id": "CP0.2a", "kind": "expr", "mistakes": {"a64fd4f04791dcddaec312410106a1efd0537f0f36c0a6e14336e65664c7dffc": "un signe moins devant une parenthèse change tous les signes qu'elle contient", "ff5549ca7dfeaf9aefc80159780e390fb9715c763e9ba5a1d4e9da8b03590ec1": "(a − b)² = a² − 2ab + b²"}, "points": [["1/3"], ["5/7"], ["6/5"], ["-2/5"], ["9/4"]], "variables": ["x"]}
📝 Ex CP0.2a : réponse enregistrée (expr).
WBM_ANSWER {"decimals": 0, "element_hashes": ["dc72eaaff624ee882568cdac1e8fdb8c190426da8ae3a8279a4b43349152a996", "839dcb3030a847115fde7ed679bdbc9499010b93674a55f2a16a55b11805a7d2"], "hash": "29737b14661232ed25e345b41f14d243dcb467681a802b06182f42388d3d68d4", "id": "CP0.2b", "integer": true, "kind": "array", "mistakes": {"f784b31e080de13c77e1d633d21ce70488faecf8e1f4ebdeaf83c6596183fb24": "diviser les deux membres par x fait perdre une solution : factorise"}, "shape": [2]}
📝 Ex CP0

**CP0.3 — Un système 2 × 2 et sa lecture graphique**

In [5]:
wbm.record("CP0.3a", [2, 3], mistakes={"vérifie ta solution dans les deux équations, dans l'ordre [x, y]": [3, 2]})
wbm.record("CP0.3b", 3 * (-2) - 1 * 1, mistakes={"le déterminant vaut ad − bc : on soustrait le second produit": -5})
wbm.record("CP0.3c", Fraction(1, 2), kind="fraction", simplified=True, mistakes={"isole y, puis divise par son coefficient": "2", "isole y en gardant les signes : passe x de l'autre côté": "-1/2"})
wbm.record("CP0.3d", [2, -2], mistakes={"la pente est l'écart des ordonnées divisé par l'écart des abscisses": [0.5, -2], "p est l'ordonnée du point où la droite coupe l'axe vertical": [2, 1]})
wbm.record("CP0.3e", [2.2, 2.4], decimals=1, mistakes={"c'est le point commun à D1 et D2 ; prends l'équation de D3 trouvée en d)": [2, 3]})
wbm.record("CP0.3f", 3, choices=[1, 2, 3], mistakes={"compare les deux équations : l'une est-elle un multiple de l'autre ?": 1})

WBM_ANSWER {"decimals": 0, "element_hashes": ["b5d74732d2f9e7f57c1e9bc8c945ddd725d17481778619df5128c120a78f549b", "2d0b3f7cbd8b4e52f31ef9f57bd9770f9d6548d7da0dc5c48acbc9257dcfec29"], "hash": "19cbda5f67e8c5c18776dfb94a5c8aa621e30067c69b2e7709f040e458f4fb67", "id": "CP0.3a", "integer": true, "kind": "array", "mistakes": {"bf6245863ca3ef9281b64369486b7cbbcd8a92956f1ce7f349316352c25daa0f": "vérifie ta solution dans les deux équations, dans l'ordre [x, y]"}, "shape": [2]}
📝 Ex CP0.3a : réponse enregistrée (array, 0 décimale).
WBM_ANSWER {"hash": "9ed6d47810895e8b666d803e02debcede2b52d97c882fa135a62162be12ba4b4", "id": "CP0.3b", "kind": "int", "magnitude": 0, "mistakes": {"e3a6dce3e3c4583402c4ecd96042cf00e68b183b580440ef8050e3300713514f": "le déterminant vaut ad − bc : on soustrait le second produit"}, "sign": -1}
📝 Ex CP0.3b : réponse enregistrée (int).
WBM_ANSWER {"hash": "de87596723505a891ce85f0a2653957dc716cc0b3bfb2d82f1900296f2bc0269", "id": "CP0.3c", "kind": "fraction", "mistakes": {"

**CP0.4 — Second degré : racines et sommet**

In [6]:
wbm.record("CP0.4a", (-6) ** 2 - 4 * (-1) * 7, mistakes={"Δ = b² − 4ac, avec les signes de a et de c": 8})
wbm.record("CP0.4b", [-7, 1], mistakes={"le dénominateur de la formule est 2a, avec le signe de a": [-1, 7]})
wbm.record("CP0.4c", [-3, 16], mistakes={"l'abscisse du sommet est −b/(2a), avec les deux signes": [3, -20]})
wbm.record("CP0.4d", "]-7 ; 1[", kind="interval", mistakes={"a < 0 : la parabole est tournée vers le bas, f est positive entre ses racines": "]-oo ; -7[ U ]1 ; +oo[", "f(x) > 0 est strict : les racines sont exclues": "[-7 ; 1]"})
wbm.record("CP0.4e", 0, mistakes={"compare 20 à la plus grande valeur que prend f (l'ordonnée du sommet)": 2, "le nombre de solutions de f(x) = k se lit en comparant k au maximum de f": 1})
wbm.record("CP0.4f", 7, mistakes={"le sommet n'est pas dans [0 ; 2] : regarde le sens de variation de f sur cet intervalle": 16, "f est décroissante sur [0 ; 2] : sa plus grande valeur est au bord gauche": -9})

WBM_ANSWER {"hash": "dead53687b23ff4fd46b2e5c081e1b612271109c8c186df92c0a56d4d753340b", "id": "CP0.4a", "kind": "int", "magnitude": 1, "mistakes": {"f2fd5e05850b3d6c1c465522d74a633c995e5fd024b2004703ec1ea654086652": "Δ = b² − 4ac, avec les signes de a et de c"}, "sign": 1}
📝 Ex CP0.4a : réponse enregistrée (int).
WBM_ANSWER {"decimals": 0, "element_hashes": ["e2ba82587419f8e23a996f1b1c0423ce5046e99bfc1bbd4b2c50ea6d114399bf", "228bfb44d2c2eb09ec88a2a0802299aad83d4193b99abfaddea80c02275f5f7b"], "hash": "9e1c7020a248a344c4821841c95066b165fe3779a76b42f1ac0de4552bd810ac", "id": "CP0.4b", "integer": true, "kind": "array", "mistakes": {"def0968c53dce335257f6379393104b1b43ae67c62c6ac286366d3dd597c0ec3": "le dénominateur de la formule est 2a, avec le signe de a"}, "shape": [2]}
📝 Ex CP0.4b : réponse enregistrée (array, 0 décimale).
WBM_ANSWER {"decimals": 0, "element_hashes": ["714899a96f463698480cfae8516391229911c0beeeadc4ba8edd900cdda071d9", "aa1f90fd23dabc70f66636fb0e8e429c803de50e5f6a3675ee

**CP0.5 — Exponentielle et logarithme**

In [7]:
wbm.record("CP0.5a", 0)
wbm.record("CP0.5b", (sp.log(3) + 5) / 2, kind="exact", mistakes={"isole 2x avant de diviser : ajoute 5 aux deux membres": "(log(3) - 5)/2", "divise tout le membre de droite par 2, pas seulement ln 3": "log(3)/2 + 5"})
wbm.record("CP0.5c", (math.e + 3) / 4, decimals=2, mistakes={"ln(4x − 3) = 1 équivaut à 4x − 3 = e : la réciproque de ln est exp": 0.75, "isole x en gardant les signes : ajoute 3, puis divise par 4": (math.e - 3) / 4})
wbm.record("CP0.5d", 3)
wbm.record("CP0.5e", (Fraction(2) - Fraction(1, 2)) / (Fraction(2) + Fraction(1, 2)), kind="fraction", simplified=True, mistakes={"e^(−ln a) = 1/a, pas a": "0", "le numérateur de tanh est la différence, son dénominateur la somme": "5/3"})

WBM_ANSWER {"hash": "14357a6d097fb5c80778776d2795d3b63327ec6cbc68fdc9877296f39aae252f", "id": "CP0.5a", "kind": "int", "magnitude": null, "sign": 0}
📝 Ex CP0.5a : réponse enregistrée (int).
WBM_ANSWER {"hash": "c494b408a0374bd9bcb830c9085ac35ed1b7a872f42079bc5856617d4ffdafc0", "id": "CP0.5b", "kind": "exact", "mistakes": {"77fde94788ca3aafc78c9585242d7da3de4bfab4ae7796e5333c11330b339840": "divise tout le membre de droite par 2, pas seulement ln 3", "964e0a16f166fbbf37bc910ebb069219df8bdbb5cb7209d19b5d09a2a45a94ff": "isole 2x avant de diviser : ajoute 5 aux deux membres"}, "rounded": {"1": "6476a4befe2c5bd7619e7e92a33516ce705de498f1b932aaeaf4a072db692f0d", "2": "14e97f077754be04ad50e736a59414d3c8f5d071d1325c76c09328f4ace4bfcf", "3": "1db823c4a9a93eb152897fae5d24474d531891dff82eea10ee3c20aa8110108a", "4": "6bd8494dcf643c59d739032df34b14c196f4d419b8dcbb2a260e594b8f8a148a", "5": "d6f3621e1f847c65da43f08f70c1817c1f0eb378dc5dd37216628500458fbe90", "6": "aa9bac908570188257dd96f80d45306b7f0aff

**CP0.6 — Dériver et étudier les variations**

In [8]:
wbm.record("CP0.6a", "3*x**2 - 12", kind="expr", variables=("x",), mistakes={"la dérivée d'une constante est nulle": "3*x**2 - 7"})
wbm.record("CP0.6b", [-2, 2], mistakes={"un carré égal à 4 : deux solutions, opposées": [2]})
wbm.record("CP0.6c", [21, -11], mistakes={"l'ordre demandé : le maximum local d'abord": [-11, 21], "ce sont les abscisses ; la question demande les valeurs de f": [-2, 2]})
wbm.record("CP0.6d", "[-2 ; 2]", kind="interval", mistakes={"f' est un trinôme de coefficient positif : elle est négative entre ses racines": "]-oo ; -2] U [2 ; +oo[", "f' ne s'annule qu'aux bornes : le plus grand intervalle les contient, crochets fermés": "]-2 ; 2["})
wbm.record("CP0.6e", [-9, 3], mistakes={"la tangente : y = f'(a)(x − a) + f(a) ; développe pour lire p": [-9, -6]})
wbm.record("CP0.6f", "(2*x - x**2)*exp(-x)", kind="expr", variables=("x",), mistakes={"la dérivée d'un produit est u'v + uv', pas u'v'": "-2*x*exp(-x)", "la dérivée de e^(−x) est −e^(−x) (règle de la chaîne)": "(2*x + x**2)*exp(-x)"})
wbm.record("CP0.6g", "7/(x + 3)**2", kind="expr", variables=("x",), mistakes={"(u/v)' = (u'v − uv')/v² : l'ordre des termes compte": "-7/(x + 3)**2", "la dérivée d'un quotient n'est pas le quotient des dérivées": "2"})
wbm.record("CP0.6h", 3, mistakes={"utilise le tableau de variations : compare à 0 le maximum local, le minimum local et les limites en ±∞": 2, "f(x) = 0 peut avoir une solution sur chaque intervalle où f est monotone : regarde le signe de f aux deux extrema": 1})

WBM_ANSWER {"hash": "e7f6b1b2fa1a724879db02d75b18f33f9cf97ddc994bff3fad1daaf2dcb6bdc2", "id": "CP0.6a", "kind": "expr", "mistakes": {"3597c6f7827d91e020b3fe82f390310193f1bbe5eb0cc621ad1753625bc6b686": "la dérivée d'une constante est nulle"}, "points": [["1/3"], ["5/7"], ["6/5"], ["-2/5"], ["9/4"]], "variables": ["x"]}
📝 Ex CP0.6a : réponse enregistrée (expr).
WBM_ANSWER {"decimals": 0, "element_hashes": ["452fbe3e9e01f72060c3ebb7b57863c1ef63e786ff903bb25422c802f86adcae", "f72a1f0f0d06bdae925e1fd2a884433270590b1338fdf42070b58ec7aeb1f7f8"], "hash": "7b16630bcba228589190a9bff90eecc977fa178fba9684c8e6f6e85945c45dda", "id": "CP0.6b", "integer": true, "kind": "array", "mistakes": {"087c177b9df10b8e46f56c4490d243bdf5a537357aeadc1f6906d6b421d6199d": "un carré égal à 4 : deux solutions, opposées"}, "shape": [2]}
📝 Ex CP0.6b : réponse enregistrée (array, 0 décimale).
WBM_ANSWER {"decimals": 0, "element_hashes": ["9dc63b8d7e91266b3a5b4489b6ec859f81a9cdd7aee38f574f6b1d980b6ba55e", "0504aa3c7ccb5ee

**CP0.7 — Un arbre de probabilités et une espérance**

In [9]:
wbm.record("CP0.7a", P_R, decimals=3, mistakes={"multiplie les probabilités le long de chaque branche, puis additionne": LATE_C + LATE_V, "pondère chaque taux de retard par la part des colis concernés": (LATE_C + LATE_V) / 2})
wbm.record("CP0.7b", P_V * LATE_V, decimals=3, mistakes={"P(V ∩ R) = P(V) × P(R sachant V) : n'oublie pas la première branche": LATE_V, "P(V ∩ R) est la probabilité d'être à vélo ET en retard, pas une probabilité sachant R": P_V * LATE_V / P_R})
wbm.record("CP0.7c", 1 - (1 - P_R) ** 2, decimals=4, mistakes={"les probabilités ne s'additionnent pas ici : passe par l'événement contraire": 2 * P_R, "c'est la probabilité que les deux colis soient en retard": P_R ** 2, "« au moins un » compte aussi le cas où les deux colis sont en retard": 2 * P_R * (1 - P_R)})
wbm.record("CP0.7d", E_X, decimals=2, mistakes={"pondère chaque gain par sa probabilité": (2 + (-10)) / 2})
wbm.record("CP0.7e", V_X, decimals=2, mistakes={"V(X) = E[X²] − E[X]² : retranche le carré de l'espérance": V_X + E_X ** 2, "V(X) = E[X²] − E[X]² : c'est le CARRÉ de l'espérance qu'on retranche": V_X + E_X ** 2 - E_X, "c'est l'écart-type ; la variance est son carré": math.sqrt(V_X)})
wbm.record("CP0.7f", 2 * E_X - 1, decimals=2, mistakes={"E[aX + b] = a E[X] + b : n'oublie pas la constante": 2 * E_X})

WBM_ANSWER {"coarse_any": {"1": ["b0af3fe77a894843c30315a044e7b5a9f548efa603aefa7c7ab99dead70ac7c3"]}, "decimals": 3, "hash": "34e8292e1460d0cda652cbe07fa936aa8fad8b000002acc0f541cafdb6fa513f", "hash_coarse": "5d0c1efaa1e71ad0dde23bb3f1a1de49098f94191f235fa08ff4a3f73879a357", "id": "CP0.7a", "kind": "float", "magnitude": -2, "mistakes": {"cc19d8b84e4864c544b8c319dcd50663205410252f94570d3bb2864e907ef83c": "multiplie les probabilités le long de chaque branche, puis additionne", "f7f1f2d7c41e2cb547c8b9a5dd3f27fbc2c6d6f37845157c3453c7c6a7bc2eff": "pondère chaque taux de retard par la part des colis concernés"}, "sign": 1}
📝 Ex CP0.7a : réponse enregistrée (float, 3 décimales).
WBM_ANSWER {"decimals": 3, "hash": "50c6c0e838e2959e3a1c00bedacf6d328f51de7a33ca685bdbe50dbb57fa37f9", "hash_coarse": "d2e8f1540934d3cde6e34478e4d7b537273f96133652aa3195dbf5d8fdbfe9bd", "id": "CP0.7b", "kind": "float", "magnitude": -2, "mistakes": {"5d2e3a637bcf99ad85801fa1ead2cf7a44c62cf3891b2a9503c4d99c73e41c3f": "

**CP0.8 — Vecteurs, produit scalaire, angle**

In [10]:
wbm.record("CP0.8a", [3 - (-3), -3 - 1], mistakes={"les coordonnées d'un vecteur : arrivée moins départ": [-6, 4]})
wbm.record("CP0.8b", math.sqrt(6 ** 2 + 4 ** 2), decimals=2, mistakes={"la distance est la racine carrée de la somme des carrés des écarts": 52, "élève les écarts au carré avant de les additionner": 10})
wbm.record("CP0.8c", 6 * 3 + (-4) * 2, mistakes={"multiplie les coordonnées de même rang, avec leurs signes": 26})
wbm.record("CP0.8d", Fraction(10, 26), kind="fraction", simplified=True, mistakes={"divise le produit scalaire par le produit des DEUX normes": "10/13"})
wbm.record("CP0.8e", math.degrees(math.acos(5 / 13)), decimals=1)

WBM_ANSWER {"decimals": 0, "element_hashes": ["ad23685a35bb1e02c5185eb811902c619832e3abf428093288fa0010c99330eb", "a0b70cfb0e7cb565b46c053b40a5255a01ae6965e1e0e5d65e658e854d17c10c"], "hash": "8dddcf7175221d148921a6b26b8d9e446a47a732691fc3dbe54bf1ffb92c8c94", "id": "CP0.8a", "integer": true, "kind": "array", "mistakes": {"d2e342cf1ea6735281895601d894d94f23d9feafea32e775814d018270ff8778": "les coordonnées d'un vecteur : arrivée moins départ"}, "shape": [2]}
📝 Ex CP0.8a : réponse enregistrée (array, 0 décimale).
WBM_ANSWER {"coarse_any": {"0": ["6109e0290a2868fe32b09a8a96037c7183b570f272a821bfc9a0e9c344aaf499"]}, "decimals": 2, "hash": "b98374756ebdcb78740775293cde6758d411870969c6ee9e4ba24e4e3ed58e2a", "hash_coarse": "59669621725b78bd48b5cf2826d44dc46b59dff3fac16eb85ccdb79f75670947", "id": "CP0.8b", "kind": "float", "magnitude": 0, "mistakes": {"8920e1fd7f6f924f9f91906545d6349d36d84555a271ec01eda71afb970a7b60": "la distance est la racine carrée de la somme des carrés des écarts", "ac95494

**CP0.10 — Corriger une copie : cinq erreurs (les lettres seulement)**

In [11]:
wbm.record("CP0.10a", "BCEFH", mistakes={"il manque une des lignes fausses : il y en a exactement cinq": {'BCEF', 'BCEH', 'BCFH', 'BEFH', 'CEFH'}, "une des lignes citées est juste : relis-les une par une, avec la règle qu'elle utilise": {'ABCEFH', 'BCDEFH', 'BCEFGH'}, "une ligne juste est citée à la place d'une ligne fausse": {'ABCEF', 'ABCEH', 'ABCFH', 'ABEFH', 'ACEFH', 'BCDEF', 'BCDEH', 'BCDFH', 'BCEFG', 'BCEGH', 'BCFGH', 'BDEFH', 'BEFGH', 'CDEFH', 'CEFGH'}})

WBM_ANSWER {"hash": "3476f2e0be465a8eda11a6c13fad38ae1556283bb0f871e675929e4e5c597709", "id": "CP0.10a", "kind": "str", "mistakes": {"089530dcd307f3195388f427c1dd11352b3631c35bd20b942e455635410e2ba6": "une des lignes citées est juste : relis-les une par une, avec la règle qu'elle utilise", "098c81563af90c46a33dc6a710ab1bbe6b0555626051371f0319f3882757da0b": "une ligne juste est citée à la place d'une ligne fausse", "3c394c9306247202edf9bb72eb9f81e7cf39d386d1d9c1ce45cc0687ba9bd0c4": "une des lignes citées est juste : relis-les une par une, avec la règle qu'elle utilise", "4cab8075933b670180fda1019ef5f085091c876c4c53c6ebe96fb370fc27c860": "il manque une des lignes fausses : il y en a exactement cinq", "63586a0cf5caedc7ee09ef9fa4851c723ec2c657c516c34bf906c26a8e2f2a3d": "il manque une des lignes fausses : il y en a exactement cinq", "7262f88757aa7e5cfacd62f30ea064930517f7f041eaba9f37bee0078cd656c5": "il manque une des lignes fausses : il y en a exactement cinq", "76e9d9e2475c3e961e882b426ec

**CP0.11 — Une somme écrite avec Σ et une somme géométrique**

In [12]:
wbm.record("CP0.11a", sum(k ** 2 - 1 for k in range(1, 5)), mistakes={"le − 1 est dans chaque terme : il compte autant de fois qu'il y a de termes": 29})
wbm.record("CP0.11b", 5 * (2 ** 5 - 1) // (2 - 1), mistakes={"compte les termes : k va de 0 à 4": 75})
wbm.record("CP0.11c", (12 - 4 + 1) * 3, mistakes={"de k = 4 à k = 12 : compte les termes, bornes comprises": 24})
wbm.record("CP0.11d", 1 / (1 - Fraction(2, 5)), kind="fraction", simplified=True, mistakes={"le premier terme (k = 0) vaut 1 : la somme vaut 1/(1 − q)": "2/3"})
wbm.record("CP0.11e", math.prod(2 ** k for k in range(1, 5)), mistakes={"Π multiplie les termes ; c'est Σ qui les additionne": 2 + 4 + 8 + 16, "un produit de puissances d'un même nombre additionne les exposants, il ne les multiplie pas": 2 ** 24})

WBM_ANSWER {"hash": "3fda0c4e5000d28c18273a5816215abc7aa9abb07cd91af52cd9b0a76abdd7b1", "id": "CP0.11a", "kind": "int", "magnitude": 1, "mistakes": {"a54f5a7acc5acd050e846ecd0e0fa1606e888faca222d1bb117821110250b20c": "le − 1 est dans chaque terme : il compte autant de fois qu'il y a de termes"}, "sign": 1}
📝 Ex CP0.11a : réponse enregistrée (int).
WBM_ANSWER {"hash": "b49beade36bcd1b4d40bc9660edc7fbf790b701c22e1732261fbfd4053bdeaf6", "id": "CP0.11b", "kind": "int", "magnitude": 2, "mistakes": {"6519287406488c47ca81d0e0c28f146796888e12e08745ab644785c4ca411872": "compte les termes : k va de 0 à 4"}, "sign": 1}
📝 Ex CP0.11b : réponse enregistrée (int).
WBM_ANSWER {"hash": "4e037a53a9fba642a2c553b97f680bca4b007724cbbfe1aab89ffbc011286e3a", "id": "CP0.11c", "kind": "int", "magnitude": 1, "mistakes": {"1be26743d75ef213e0e246ead7debc11709f1006562bd0af1bbc3740a86bf6ee": "de k = 4 à k = 12 : compte les termes, bornes comprises"}, "sign": 1}
📝 Ex CP0.11c : réponse enregistrée (int).
WBM_ANSWER {

**CP0.12 — Problème de synthèse : étudier une fonction de coût donnée par une formule**

In [13]:
wbm.record("CP0.12a", round(math.exp(0) - 2 * 0), mistakes={"e⁰ = 1": 0})
wbm.record("CP0.12b", "exp(w) - 2", kind="expr", variables=("w",), mistakes={"dérive aussi le terme −2w": "exp(w) - 2*w"})
wbm.record("CP0.12c", sp.log(2), kind="exact", mistakes={"eʷ = 2 : on prend le logarithme népérien des deux membres": "2", "la réciproque de l'exponentielle est le logarithme népérien": "exp(2)"})
wbm.record("CP0.12d", 2 - 2 * math.log(2), decimals=3, mistakes={"e^(ln a) = a : remplace w par w* dans les deux termes": 1 - 2 * math.log(2)})
wbm.record("CP0.12e", "]-oo ; log(2)]", kind="interval", mistakes={"L' est négative avant w* : L décroît, puis croît": "[log(2) ; +oo[", "L' ne s'annule qu'en w* : le plus grand intervalle le contient, crochet fermé": "]-oo ; log(2)["})
wbm.record("CP0.12f", math.inf, decimals=2, mistakes={"croissances comparées : eʷ l'emporte sur 2w": -math.inf})
wbm.record("CP0.12g", [-1, 1], mistakes={"la tangente en 0 : y = L'(0) w + L(0)": [-1, 0]})
wbm.record("CP0.12h", True, mistakes={"compare à 0 la plus petite valeur de L, trouvée en d)": False})

WBM_ANSWER {"hash": "20777e0e6b9e83af1db8ae7053401fd9f14345cfcbad7d2169566bceecbb4677", "id": "CP0.12a", "kind": "int", "magnitude": 0, "mistakes": {"f5fdeb61600af5bdadbff9f22695d8bb83aceaf3927326786e44f1c72e97e24d": "e⁰ = 1"}, "sign": 1}
📝 Ex CP0.12a : réponse enregistrée (int).
WBM_ANSWER {"hash": "1f50cdbd547931ec99e9490a636895f0dc375ce1b903cc0f1527a616e84fa8b1", "id": "CP0.12b", "kind": "expr", "mistakes": {"4c2648667930d2680c27d37fe0299c4a3ec0a1564f07d5fb48cd66c9840bda9d": "dérive aussi le terme −2w"}, "points": [["1/3"], ["5/7"], ["6/5"], ["-2/5"], ["9/4"]], "variables": ["w"]}
📝 Ex CP0.12b : réponse enregistrée (expr).
WBM_ANSWER {"hash": "0b174226076fa3e131627efa045f3bb8c9f86fd10faf9e5b757dd2435f7d396a", "id": "CP0.12c", "kind": "exact", "mistakes": {"50ea4f1d7bebddd09d845eac9dc8f6e517de3f9705c0e58b487f466dcc7a4fa7": "la réciproque de l'exponentielle est le logarithme népérien", "67925f050bf18ca5f301314ffc423a017c069ee57f2528fae17bf48dbc1e9abe": "eʷ = 2 : on prend le logarithme

WBM_ANSWER {"hash": "58116d85f79b052cb75965af568996be3a388084c480a01429a8326bb11c3fae", "id": "CP0.12e", "kind": "interval", "mistakes": {"485fae5218d994325ab8f702d78b183e248684c0addcbaf557b9c2acc4cba2b0": "L' est négative avant w* : L décroît, puis croît", "799b983120138c851b43e0a6733f50b2be2194e3df2971b22308b11f62bcad06": "L' ne s'annule qu'en w* : le plus grand intervalle le contient, crochet fermé"}, "rounded": {"1": "ebbd49af97e83d0389d40eb38098e2b6fa0ddb25ea9a174e91f7df70f50ac60c", "2": "01d0efb883d634b3aad6b11d39b5cbb6123d9a185d84be2a14c97c34fc2af542", "3": "303afbd190345141398995b32dfa02696c71b365331ab2563db840ba8268d7b0", "4": "37297cbe92d670318c82c496a4ecbf22aae56f8edd6f4e43b6df1ce49ebcbca1", "5": "32af7fefd089b8709cfe5c63ee6f32444001b4444d9ac9fa8f4843627ed7baa4"}}
📝 Ex CP0.12e : réponse enregistrée (interval).
WBM_ANSWER {"coarse_any": {"0": ["4b2bdb9f04fb97967df65365fc666e1db5b7ce00fcab3e77010fa18bb9313e1a"]}, "decimals": 2, "hash": "4b2bdb9f04fb97967df65365fc666e1db5b7ce00